## Run initial SUMMA model tests ##

In [ ]:
from pathlib import Path

from snowda.config.loader import load_config
from snowda.workflows.run_summa_da import run_summa_da

config_path = Path("../runs/examples/ross_summa_esp_da.yaml").resolve()
config = load_config(config_path)
run_dir = run_summa_da(config)
print("SUMMA run dir:", run_dir)

In [ ]:
# SUMMA DA (particle filter, then ESP)
from snowda.workflows.run_summa_da import run_summa_da
config_path = Path("../runs/examples/ross_summa_da.yaml").resolve()
run_dir = run_summa_da(config_path)
print("DA run dir:", run_dir)

## Historic traces, assimilation states, and forecasts

Gray lines are every historic ensemble member. Dotted lines mark assimilation times, and black dots are the member states at those times. Red stars on the SWE plot are the assimilated basin SWE. Colored bands are the ESP forecast from that date: a light p0–p100 range, then the p10–p90 and p25–p75 envelopes used by the ESP streamflow plot, and the median. Historic streamflow is SUMMA runoff converted to m3 s-1 with the basin area so it matches the routed ESP traces. The orange line is the gauge.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from snowda.io.da_traces import load_da_streamflow, load_da_swe
from snowda.io.streamflow import read_streamflow_observations
from snowda.obs_qm.seasonal import water_year
from snowda.viz.summa_da import plot_da_traces

da_run_dir = Path("/home/x-dcasson/GitRepos/snowda/outputs/ross_summa_esp_da_sir/20261120T042248Z)

print("Plotting", da_run_dir)

figure_dir = da_run_dir / "figures"
figure_dir.mkdir(exist_ok=True)


def _trace_times(traces) -> pd.DatetimeIndex:
    pieces = [
        pd.DatetimeIndex(pd.to_datetime(segment["time"].values))
        for member in traces.historic
        for segment in member.segments
    ]
    pieces.extend(
        pd.DatetimeIndex(pd.to_datetime(forecast.values["time"].values))
        for forecast in traces.forecasts
    )
    if traces.states:
        pieces.append(pd.DatetimeIndex([state.time for state in traces.states]))
    if not pieces:
        return pd.DatetimeIndex([])
    times = pieces[0]
    for piece in pieces[1:]:
        times = times.union(piece)
    return times


swe = load_da_swe(da_run_dir)
eval_cfg = yaml.safe_load(
    Path("../runs/examples/ross_evaluate_esp.yaml").read_text()
)
obs = read_streamflow_observations(eval_cfg["obs_csv"])
flow = load_da_streamflow(da_run_dir)

# Water year starts 1 October: 1 Oct 2015 belongs to water year 2016.
times = _trace_times(swe).union(_trace_times(flow))
water_years = sorted({int(year) for year in water_year(times)})
wy_dir = figure_dir / "water_years"
wy_dir.mkdir(exist_ok=True)

for wy in water_years:
    start = pd.Timestamp(year=wy - 1, month=10, day=1)
    end = pd.Timestamp(year=wy, month=10, day=1) - pd.Timedelta(seconds=1)
    fig, _ax = plot_da_traces(
        swe,
        start=start,
        end=end,
        title=f"Ross SWE: water year {wy}",
    )
    fig.savefig(wy_dir / f"da_swe_wy{wy}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    fig, _ax = plot_da_traces(
        flow,
        obs,
        start=start,
        end=end,
        title=f"Ross streamflow: water year {wy}",
    )
    fig.savefig(wy_dir / f"da_streamflow_wy{wy}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Wrote {len(water_years)} water years to {wy_dir}")